# Lab: Tokenisers, scaling laws and compute

**Session 2**

| Part | Topic |
| ---- | ----- |
| 1 | The causal-LM objective: loss, perplexity, shift by one |
| 2 | BPE from scratch on a toy corpus |
| 3 | Byte-level BPE tokenisers on WikiText |
| 4 | Sizing worksheet: FLOPs, GPU-hours, euros |

Every exercise is one to three lines: replace each `____` and run the check cell below it.

## Lab rules

- Replace every `____`, then run the check cell that follows. Do not continue until the check passes.
- Evaluate tokenisers on the **validation** split only, never on the text they were trained on.
- Count **tokens of your tokeniser**, never words or documents, when you compute `D`.
- Every number in the worksheet comes with its **assumptions**.

### Notation (same as the slides)

| Symbol    | Meaning                                                   |
| --------- | --------------------------------------------------------- |
| `V`       | vocabulary size                                           |
| `T`       | context length (tokens per sequence)                      |
| `d_model` | model width (size of each token vector), Session 1's `C`  |
| `L`       | number of transformer layers                              |
| `N`       | number of model parameters                                |
| `D`       | number of training tokens                                 |
| `C`       | training compute, in FLOPs                                |

## Installation

Everything runs on a laptop CPU (no GPU needed). The longest step is the WikiText download (about 300 MB, once).

With `uv`, from the `sessions/` folder (one environment for all the labs):

```bash
uv sync
```

Then select the `.venv` kernel in your notebook editor.

In [ ]:
import math
import time
from collections import Counter
from pathlib import Path

import torch
import torch.nn.functional as F
import tokenizers
import datasets
import transformers

torch.manual_seed(0)
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

print(
    f"torch {torch.__version__}, tokenizers {tokenizers.__version__}, "
    f"datasets {datasets.__version__}, transformers {transformers.__version__}"
)


def check_close(actual, expected, name, rel=1e-3):
    assert math.isclose(
        actual, expected, rel_tol=rel
    ), f"{name}: expected about {expected:,.4g}, got {actual:,.4g}"
    print(f"OK: {name}: {actual:,.4g}")

# Part 1: The causal-LM objective

A language model is trained to maximise the probability of the real next token. For one position, the loss is

$$\text{loss} = -\ln p(\text{true next token})$$

and over a sequence we average these losses. Perplexity is $e^{\text{mean loss}}$: the number of tokens the model is "hesitating between".

### Exercise 1.1: Loss and perplexity

Context: _"The cat sat on the"_. The model predicts `{" mat": 0.60, " sofa": 0.30, " dog": 0.09, " France": 0.01}`.

Complete `token_loss` (one line). `perplexity` is provided.

In [ ]:
def token_loss(p_true: float) -> float:
    # TODO: the loss when the model gives probability p_true to the true token: -ln(p). Hint: math.log
    return ____


def perplexity(mean_loss: float) -> float:
    return math.exp(mean_loss)  # e to the power of the average loss

In [ ]:
next_token_probs = {" mat": 0.60, " sofa": 0.30, " dog": 0.09, " France": 0.01}
for token, p in next_token_probs.items():
    print(f"true token = {token!r:<10} p = {p:<5} loss = {token_loss(p):.2f}")

check_close(token_loss(0.60), 0.5108, "loss for ' mat'")
check_close(token_loss(0.01), 4.6052, "loss for ' France'")
check_close(perplexity(math.log(17)), 17, "perplexity of loss ln(17)")

**Question.** A model gives the probabilities `[0.20, 0.50, 0.90, 0.05]` to the 4 true tokens of a sentence. Run the cell below. Which token costs the most, and why does a single confident mistake weigh so much?

In [ ]:
sentence_probs = [0.20, 0.50, 0.90, 0.05]

losses = [token_loss(p) for p in sentence_probs]
mean_loss = sum(losses) / len(losses)
print("losses:", [round(x, 2) for x in losses])
print(f"mean loss = {mean_loss:.3f}, perplexity = {perplexity(mean_loss):.2f}")

### Exercise 1.2: Shift by one

A training sequence of token IDs gives **one prediction per position**: the input is the sequence without its last token, the target is the sequence without its first token.

In [ ]:
def make_lm_pairs(ids: list[int]) -> tuple[list[int], list[int]]:
    # TODO: the targets: the sequence without its first token
    return ids[:-1], ____

In [ ]:
BOS, EOS = 0, 1
ids = [BOS, 464, 3797, 3332, 319, 262, 2603, EOS]  # <bos> The cat sat on the mat <eos>
inputs, targets = make_lm_pairs(ids)
assert inputs == [0, 464, 3797, 3332, 319, 262, 2603]
assert targets == [464, 3797, 3332, 319, 262, 2603, 1]
print(f"OK: {len(inputs)} next-token predictions from one sequence of {len(ids)} tokens")

### Exercise 1.3: The `ln V` sanity check

Before training, a model with small random weights predicts an almost **uniform** distribution over the vocabulary. Its loss should therefore be close to `ln V`. This is the first number to check when you launch a training run.

Run the cell, then answer: what initial loss do you expect for GPT-2 (`V = 50,257`)? For Llama 3 (`V = 128,256`)?

In [ ]:
V = 8_000
B, T = 4, 16
logits = torch.randn(B, T, V) * 0.02  # what a freshly initialised LM head produces
targets = torch.randint(0, V, (B, T))

loss = F.cross_entropy(logits.reshape(-1, V), targets.reshape(-1))
print(f"initial loss = {loss.item():.3f}, ln V = {math.log(V):.3f}")
print(f"GPT-2: ln V = {math.log(50_257):.2f}, Llama 3: ln V = {math.log(128_256):.2f}")

# Part 2: BPE from scratch

BPE training repeats three steps until the vocabulary has the target size:

1. count every **adjacent pair** of symbols, weighted by word frequency
2. pick the **most frequent** pair
3. **merge** it everywhere into a new symbol

We work on the toy corpus of the slides: `cat ×8, cats ×4, hat ×5, that ×6, chat ×3`.

### Provided: `merge`

`merge` replaces every occurrence of `pair` in a tuple of symbols by the concatenated symbol. It scans left to right; once two symbols are merged, both are skipped. Run the cell and its checks.

In [ ]:
def merge(symbols: tuple[str, ...], pair: tuple[str, str]) -> tuple[str, ...]:
    out, i = [], 0
    while i < len(symbols):
        if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
            out.append(symbols[i] + symbols[i + 1])
            i += 2
        else:
            out.append(symbols[i])
            i += 1
    return tuple(out)


assert merge(("c", "a", "t"), ("a", "t")) == ("c", "at")
assert merge(("t", "h", "a", "t"), ("a", "t")) == ("t", "h", "at")
assert merge(("a", "a", "a"), ("a", "a")) == ("aa", "a")  # left to right, no overlap
assert merge(("h", "at"), ("c", "at")) == ("h", "at")  # nothing to merge
print("OK: merge")

### Exercise 2.1: `train_bpe`

`count_pairs` is provided. In `train_bpe`, write one line: pick the most frequent pair.

In [ ]:
def count_pairs(words: dict[tuple[str, ...], int]) -> Counter:
    pairs = Counter()
    for symbols, count in words.items():
        for pair in zip(symbols, symbols[1:]):
            pairs[pair] += count  # every occurrence of the word counts
    return pairs


def train_bpe(word_counts: dict[str, int], n_merges: int) -> list[tuple[str, str]]:
    words = {tuple(w): c for w, c in word_counts.items()}
    merges = []
    for _ in range(n_merges):
        pairs = count_pairs(words)
        if not pairs:
            break
        best = ____  # TODO: the most frequent pair. Hint: pairs.most_common(1)[0][0]
        merges.append(best)
        words = {merge(symbols, best): c for symbols, c in words.items()}
    return merges

In [ ]:
toy_corpus = {"cat": 8, "cats": 4, "hat": 5, "that": 6, "chat": 3}

toy_merges = train_bpe(toy_corpus, n_merges=5)
assert toy_merges[:4] == [
    ("a", "t"),
    ("h", "at"),
    ("c", "at"),
    ("t", "hat"),
], toy_merges
print("OK: merges:", toy_merges)

### Exercise 2.2: Encoding new words

To encode a word, start from its characters and apply the learned merges **in the order they were learned**.

In [ ]:
def encode_word(word: str, merges: list[tuple[str, str]]) -> list[str]:
    symbols = tuple(word)
    for pair in merges:
        symbols = ____  # TODO: apply this merge to symbols. Hint: merge(symbols, pair)
    return list(symbols)

In [ ]:
assert encode_word("chats", toy_merges) == ["c", "hat", "s"]
assert encode_word("that", toy_merges) == ["that"]
print("OK: chats ->", encode_word("chats", toy_merges))
for w in ["hats", "cathat", "bat", "act"]:
    print(f"{w:<8} -> {encode_word(w, toy_merges)}")

**Questions.** Run the cell below, then answer in one sentence each.

1. `"bat"` contains a `b` that never appeared in the corpus. What would a real tokeniser do with an unknown character? How does **byte-level** BPE avoid the problem?
2. In `{"low": 5, "lower": 2, "lowest": 3}`, two pairs have the **same count** at the start. What decides which one is merged first? Is the result deterministic?

In [ ]:
low_corpus = {"low": 5, "lower": 2, "lowest": 3}
start_words = {tuple(w): c for w, c in low_corpus.items()}
print("pair counts at the start:", count_pairs(start_words).most_common(4))
print("3 merges:", train_bpe(low_corpus, 3))

# Part 3: Byte-level BPE tokenisers on WikiText

[WikiText-103](https://huggingface.co/datasets/Salesforce/wikitext) contains about 100M words from 28,475 good-quality Wikipedia articles. We use a **slice** of the train split to stay fast on CPU, and keep the **validation** split for evaluation.

The `tokenizers` library (written in Rust) implements the same algorithm as Part 2, with production details:

- a **byte-level** initial alphabet (256 symbols): any string can be encoded, no `<unk>`
- a **pre-tokeniser** that splits on spaces and punctuation, so merges never cross words (spaces are attached to the next word and displayed as `Ġ`)
- **special tokens** added by hand

In [ ]:
wikitext = datasets.load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")
print(wikitext)

TRAIN_SLICE = 50_000  # rows of the train split used to train a tokeniser
train_texts = [
    t for t in wikitext["train"].select(range(TRAIN_SLICE))["text"] if t.strip()
]
valid_texts = [t for t in wikitext["validation"]["text"] if t.strip()]

print(
    f"train slice: {len(train_texts):,} non-empty lines, {sum(len(t) for t in train_texts) / 1e6:.1f}M characters"
)
print(
    f"validation : {len(valid_texts):,} non-empty lines, {sum(len(t) for t in valid_texts) / 1e6:.1f}M characters"
)
print("\nExample line:", repr(train_texts[3][:300]))

### Exercise 3.1: The training function

The tokeniser, the pre-tokeniser, the decoder and the trainer are already built. One blank: give the trainer the text to learn from, as batches.

In [ ]:
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

SPECIAL_TOKENS = ["<|endoftext|>", "<pad>"]


def batch_iterator(texts: list[str], batch_size: int = 1_000):
    for i in range(0, len(texts), batch_size):
        yield texts[i : i + batch_size]


def train_bpe_tokenizer(texts: list[str], vocab_size: int) -> Tokenizer:
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=SPECIAL_TOKENS,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        show_progress=False,
    )
    # TODO: replace the blank by the batches of text. Hint: batch_iterator(texts)
    tok.train_from_iterator(____, trainer=trainer, length=len(texts))
    return tok

In [ ]:
small = train_bpe_tokenizer(train_texts[:2_000], vocab_size=500)
assert 256 < small.get_vocab_size() <= 500, small.get_vocab_size()
print(f"OK: trained a tokeniser with {small.get_vocab_size()} tokens")
print(small.encode("The capital of France is Paris.").tokens)

### Provided: three vocabulary sizes

By default the cell loads the three tokenisers already saved in `artifacts/` (trained on 200,000 rows). Set `LOAD_PRECOMPUTED = False` to train them yourself with your function.

In [ ]:
VOCAB_SIZES = [1_000, 8_000, 32_000]
LOAD_PRECOMPUTED = True  # False: train the three tokenisers yourself
our_tokenizers: dict[int, Tokenizer] = {}

for vocab_size in VOCAB_SIZES:
    path = ARTIFACTS / f"wikitext-bpe-{vocab_size // 1000}k.json"
    if LOAD_PRECOMPUTED and path.exists():
        our_tokenizers[vocab_size] = Tokenizer.from_file(str(path))
        print(f"V = {vocab_size:>6,}: loaded {path.name}")
    else:
        start = time.perf_counter()
        tok = train_bpe_tokenizer(train_texts, vocab_size)
        tok.save(str(path))
        our_tokenizers[vocab_size] = tok
        print(f"V = {vocab_size:>6,}: trained in {time.perf_counter() - start:5.1f}s")

In [ ]:
roundtrip_samples = [
    "The capital of France is Paris.",
    "Le chat dort sur le canapé 🐈 — c'est l'été !",
    "def f(x):\n    return x ** 2  # 2026",
]

for s in roundtrip_samples:
    print(s)
    for vocab_size, tok in our_tokenizers.items():
        print(f"  V = {vocab_size:>6,}:", [tok.decode([id]) for id in tok.encode(s).ids])
        assert tok.decode(tok.encode(s).ids) == s, f"round trip failed for {s!r}"

### Provided: the cost of a small vocabulary

Tokens per 1,000 characters on the **validation** split: fewer tokens means a shorter sequence for the same text.

In [ ]:
sample = "".join(valid_texts[:2_000])
for vocab_size, tok in our_tokenizers.items():
    n_tokens = len(tok.encode(sample).ids)
    print(f"V = {vocab_size:>6,}: {1000 * n_tokens / len(sample):6.1f} tokens per 1,000 characters")

**Question.** Which vocabulary size would you choose for a small model, and why? One sentence. Then run the cell below: it keeps the 8k tokeniser for Part 4 (change `CHOSEN_VOCAB` if you decided otherwise).

In [ ]:
CHOSEN_VOCAB = 8_000

chosen_tok = our_tokenizers[CHOSEN_VOCAB]
chosen_tok.save(str(ARTIFACTS / "wikitext-bpe-chosen.json"))
print(f"saved artifacts/wikitext-bpe-chosen.json (V = {chosen_tok.get_vocab_size():,})")

# Part 4: Sizing worksheet

Three formulas from the slides:

$$C \approx 6\,N\,D \qquad\qquad D_{\text{opt}} \approx 20\,N \qquad\qquad N_{\text{opt}} = \sqrt{C / 120}$$

and, to go from FLOPs to time:

$$\text{time} \approx \frac{C}{\#\text{GPUs} \times \text{peak FLOP/s} \times \text{MFU}}$$

### Exercise 4.1: The sizing toolkit

Four functions are provided. Complete `compute_optimal` (one blank), then run the check cell, which reproduces the worked examples of the slides.

In [ ]:
A100_PEAK_FLOPS = 312e12  # BF16 dense, per GPU
H100_PEAK_FLOPS = 989e12
TOKENS_PER_PARAM = 20


def training_flops(n_params: float, n_tokens: float) -> float:
    return 6 * n_params * n_tokens


def chinchilla_tokens(n_params: float, ratio: float = TOKENS_PER_PARAM) -> float:
    return ratio * n_params


def chinchilla_params(n_tokens: float, ratio: float = TOKENS_PER_PARAM) -> float:
    return n_tokens / ratio


def compute_optimal(
    flops: float, ratio: float = TOKENS_PER_PARAM
) -> tuple[float, float]:
    # TODO: C = 6 * N * (ratio * N), so N = sqrt(C / (6 * ratio)). Hint: math.sqrt
    n_opt = ____
    return n_opt, ratio * n_opt


def training_hours(
    flops: float,
    n_gpus: int = 1,
    peak_flops: float = A100_PEAK_FLOPS,
    mfu: float = 0.40,
) -> float:
    seconds = flops / (n_gpus * peak_flops * mfu)
    return seconds / 3600

In [ ]:
# The worked examples of the slides
check_close(chinchilla_tokens(125e6), 2.5e9, "D for a 125M model")
check_close(chinchilla_params(600e6), 30e6, "N for 600M tokens")
check_close(training_flops(125e6, 2.5e9), 1.875e18, "C for 125M params x 2.5B tokens")
check_close(training_hours(1.875e18), 4.17, "hours on one A100 at 40% MFU", rel=1e-2)
n_opt, d_opt = compute_optimal(5.76e23)
check_close(n_opt, 70e9, "N_opt at Gopher's budget (Chinchilla: 70B)", rel=0.05)
check_close(d_opt, 1.4e12, "D_opt at Gopher's budget (Chinchilla: 1.4T tokens)", rel=0.05)
check_close(training_flops(405e9, 15.6e12), 3.79e25, "Llama 3.1 405B compute", rel=1e-2)

### Exercise 4.2: Data-constrained: how big a model does WikiText-103 support?

Count the tokens of the WikiText-103 train split with your chosen tokeniser, then apply the 20:1 rule. The counting is provided and uses 10% of the rows to stay fast (set `FRACTION = 1.0` for the exact count). Write the line that gives the model size.

In [ ]:
FRACTION = 0.1  # share of the train split that is encoded; the count is scaled up
full_train = wikitext["train"]["text"]
subset = full_train[: int(len(full_train) * FRACTION)]

start = time.perf_counter()
counted = sum(
    len(e.ids)
    for batch in batch_iterator(subset, 10_000)
    for e in chosen_tok.encode_batch(batch)
)
wikitext_tokens = counted / FRACTION
print(
    f"WikiText-103 train: {wikitext_tokens / 1e6:,.1f}M tokens (counted in {time.perf_counter() - start:.0f}s on {FRACTION:.0%} of the rows)"
)

# TODO: the model size that this many tokens supports. Hint: chinchilla_params(...)
n_data_constrained = ____
c_data_constrained = training_flops(n_data_constrained, wikitext_tokens)
print(
    f"Chinchilla-optimal N = {n_data_constrained / 1e6:.1f}M parameters, C = {c_data_constrained:.2e} FLOPs, "
    f"{training_hours(c_data_constrained):.2f} h on one A100"
)

### Exercise 4.3: The bill

Prices: **€2 per A100-hour** and **€2.50 per H100-hour** (typical cloud prices). The compute and the hours are provided, and so is the first cost. Write the second: hours × number of GPUs × price per GPU-hour.

Two reference runs are printed as well: one A100 for a day, and the data-constrained model trained **10× longer** than Chinchilla (200 tokens per parameter, about 10 epochs of WikiText).

In [ ]:
EUR_PER_A100_HOUR = 2.0
EUR_PER_H100_HOUR = 2.5

hours_data_constrained = training_hours(c_data_constrained)
c_7b = training_flops(7e9, chinchilla_tokens(7e9))  # Chinchilla-optimal 7B model
hours_7b = training_hours(c_7b, n_gpus=1_000, peak_flops=H100_PEAK_FLOPS)

cost_data_constrained = hours_data_constrained * 1 * EUR_PER_A100_HOUR  # 1 A100
# TODO: the 7B run: hours x number of GPUs x price per GPU-hour (1,000 H100)
cost_7b = ____

cost_one_gpu_day = 24 * EUR_PER_A100_HOUR
cost_overtrained = (
    training_hours(training_flops(n_data_constrained, 200 * n_data_constrained))
    * EUR_PER_A100_HOUR
)

print(f"1. data-constrained run : €{cost_data_constrained:,.2f}")
print(f"2. one A100-day         : €{cost_one_gpu_day:,.2f}")
print(f"3. 10x over-trained     : €{cost_overtrained:,.2f}")
print(
    f"4. Chinchilla 7B        : C = {c_7b:.2e} FLOPs, {hours_7b:.1f} hours on 1,000 H100, €{cost_7b:,.0f}"
)

**Question.** Why is the real cost of training a 7B model much higher than the last number?